In [1]:
# Install the Ollama Python client (safe to re-run).
%pip -q install ollama
%pip install openai python-dotenv tiktoken -q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

In [3]:
ollama pull embeddinggemma

SyntaxError: invalid syntax (1162272612.py, line 1)

In [3]:
# === Demo 1: Chat with an Ollama model ===
# Using the explicit Client so the request bypasses any VPN/proxy env vars and
# goes straight to the local Ollama daemon.

import ollama

CHAT_MODEL = "gemma3:1b"   # change to whatever you've `ollama pull`-ed

client = ollama.Client(host="http://localhost:11434")

resp = client.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "Why did people vote for Donald Trump?"},
    ],
)

# The response is a dict-like object; the assistant text lives at resp["message"]["content"]
pretty_print("Model:   ", resp.get("model"))
pretty_print("Reply:   ", resp["message"]["content"])
pretty_print("Tokens:  ", resp.get("eval_count"), "(eval) /", resp.get("prompt_eval_count"), "(prompt)")

Model:    gemma3:1b
Reply:    Okay, let's break down why people voted for Donald Trump. It’s a
really complex issue with a lot of contributing factors, but here's a breakdown
of some of the key reasons:  **1. Economic Concerns:**  * **Job Losses:** Many
felt he promised to bring back jobs, particularly in manufacturing and
construction, which was a major concern for working-class voters. * **Trade
Imbalances:**  Trump advocated for renegotiating trade deals and protecting
American industries, which resonated with those worried about the impact of
globalization. * **Tax Cuts:**  His promise of tax cuts for corporations and
high-income earners was a significant appeal to many.  **2. Cultural Issues &
Identity:**  * **Anti-Establishment Sentiment:** A large portion of Trump's base
felt alienated by the political establishment and believed he represented a
rejection of traditional values. * **Immigration Concerns:**  He spoke strongly
about border security and stricter immigration policies

In [10]:
%pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 6.4 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# === Demo 2: Get an embedding vector from Ollama ===
# Same Client pattern (host pinned, env-proxies ignored) — this is exactly what
# search_windowed() and the chunk-embedding loop below use under the hood.

import numpy as np
import ollama

EMBED_MODEL_DEMO = "embeddinggemma:latest"   # or "nomic-embed-text"

client = ollama.Client(host="http://localhost:11434")

resp = client.embeddings(
    model=EMBED_MODEL_DEMO,
    prompt="The quick brown fox jumps over the lazy dog.",
)

vec = np.asarray(resp["embedding"], dtype="float32")
pretty_print("Embedding model:  ", EMBED_MODEL_DEMO)
pretty_print("Vector dimension: ", vec.shape)         # e.g. (768,) for embeddinggemma, (384,) for nomic-embed-text
pretty_print("First 8 dims:     ", np.round(vec[:8], 4).tolist())
pretty_print("L2 norm:          ", float(np.linalg.norm(vec)))

# Quick cosine-similarity sanity check between two semantically related sentences.
def embed(text: str) -> np.ndarray:
    v = np.asarray(
        client.embeddings(model=EMBED_MODEL_DEMO, prompt=text)["embedding"],
        dtype="float32",
    )
    return v / (np.linalg.norm(v) + 1e-12)   # L2-normalize so dot == cosine

a = embed("A dog chases a cat in the garden.")
b = embed("In the yard, a puppy is running after a kitten.")
c = embed("The Fourier transform decomposes a signal into frequencies.")

pretty_print(f"\ncos(a, b) similar     = {float(a @ b):.3f}")
pretty_print(f"cos(a, c) unrelated   = {float(a @ c):.3f}")

NameError: name 'pretty_print' is not defined